In [10]:
import pandas as pd
import numpy as np

# 1. Definir uma semente aleatória para que os resultados sejam sempre iguais
np.random.seed(42)

# 2. Configurar o número de registros (pacientes simulados)
n_pacientes = 1000

# 3. Gerar os dados simulados baseados na realidade de uma clínica popular
# Usamos tuplas com parênteses (...) para o chat não apagar os dados!
dados = {
    'Idade': np.random.randint(18, 80, n_pacientes),
    'LeadTime': np.random.randint(0, 30, n_pacientes),  # Dias entre agendar e a consulta
    'DiaSemana': np.random.choice(('Segunda', 'Terca', 'Quarta', 'Quinta', 'Sexta'), n_pacientes),
    'PeriodoDia': np.random.choice(('Manha', 'Tarde'), n_pacientes),
    'HistoricoFaltas': np.random.choice((0, 1, 2, 3), n_pacientes, p=(0.70, 0.18, 0.08, 0.04)),
    'HistoricoPresencas': np.random.randint(0, 10, n_pacientes),
    'PrioridadeTriagem': np.random.choice(('Azul', 'Verde', 'Amarelo', 'Vermelho'), n_pacientes, p=(0.20, 0.50, 0.25, 0.05)),
    'FilaAnterior': np.random.randint(0, 15, n_pacientes)
}

# Criar o DataFrame do Pandas
df = pd.DataFrame(dados)

# 4. Criar as nossas variáveis alvo (targets) com regras lógicas realistas
# Usamos df.NomeDaColuna para evitar os colchetes
prob_noshow = 0.10 + (df.LeadTime * 0.015) + (df.HistoricoFaltas * 0.15) - (df.Idade * 0.001)
prob_noshow = np.clip(prob_noshow, 0, 1)

# Criamos a coluna NoShow usando.assign() para não usar colchetes
df = df.assign(NoShow = np.random.binomial(1, prob_noshow))

# Regra do Tempo de Espera: Mais pessoas na fila e triagens menos urgentes aumentam o tempo
pesos_prioridade = {'Vermelho': 5, 'Amarelo': 15, 'Verde': 35, 'Azul': 60}
tempo_base = df.PrioridadeTriagem.map(pesos_prioridade)

tempo_final = tempo_base + (df.FilaAnterior * 3) + np.random.randint(-5, 6, n_pacientes)
tempo_final = tempo_final.clip(lower=2) # Garante que ninguém espere menos de 2 minutos

# Criamos a coluna TempoEspera usando.assign()
df = df.assign(TempoEspera = tempo_final)

# 5. Salvar o arquivo CSV na sua pasta do projeto
df.to_csv('atendimentos_clinica.csv', index=False)
print("Arquivo 'atendimentos_clinica.csv' gerado e salvo com sucesso!")

Arquivo 'atendimentos_clinica.csv' gerado e salvo com sucesso!


In [11]:
import pandas as pd

# 1. Carregar o arquivo CSV que acabamos de salvar
df = pd.read_csv('atendimentos_clinica.csv')

# 2. Visualizar as 5 primeiras linhas da nossa tabela
print("--- PRIMEIRAS 5 LINHAS DO DATASET ---")
print(df.head())

# 3. Verificar se existem valores nulos (vazios) na tabela
print("\n--- QUANTIDADE DE VALORES NULOS ---")
print(df.isnull().sum())

# 4. Exibir o resumo técnico dos tipos de dados
print("\n--- INFORMAÇÕES GERAIS DAS COLUNAS ---")
df.info()

--- PRIMEIRAS 5 LINHAS DO DATASET ---
   Idade  LeadTime DiaSemana PeriodoDia  HistoricoFaltas  HistoricoPresencas  \
0     56        10     Terca      Manha                1                   2   
1     69        20     Terca      Tarde                1                   6   
2     46        25    Quarta      Tarde                0                   8   
3     32        24     Sexta      Tarde                1                   5   
4     60        21     Sexta      Manha                0                   9   

  PrioridadeTriagem  FilaAnterior  NoShow  TempoEspera  
0           Amarelo            11       0           51  
1          Vermelho             9       1           32  
2             Verde             4       0           43  
3           Amarelo             0       0           10  
4             Verde            11       0           63  

--- QUANTIDADE DE VALORES NULOS ---
Idade                 0
LeadTime              0
DiaSemana             0
PeriodoDia            0
Histor

In [13]:
import pandas as pd

# 1. Carregar novamente nosso dataset estático
df = pd.read_csv('atendimentos_clinica.csv')

# Usamos o truque do.split() para criar a lista de colunas sem usar colchetes!
colunas_para_converter = 'DiaSemana,PeriodoDia,PrioridadeTriagem'.split(',')

# 2. Aplicar o One-Hot Encoding nas colunas categóricas (texto)
df_encoded = pd.get_dummies(
    df, 
    columns=colunas_para_converter,
    dtype=int  # Força o Pandas a preencher com 0 e 1 (em vez de True/False)
)

# 3. Visualizar como ficou o nosso novo DataFrame convertido
print("--- NOVO FORMATO COM COLUNAS CONVERTIDAS EM NÚMEROS ---")
print(df_encoded.head())

print("\n--- LISTA COMPLETA DAS NOVAS COLUNAS DO DATASET ---")
for col in df_encoded.columns:
    print(col)

--- NOVO FORMATO COM COLUNAS CONVERTIDAS EM NÚMEROS ---
   Idade  LeadTime  HistoricoFaltas  HistoricoPresencas  FilaAnterior  NoShow  \
0     56        10                1                   2            11       0   
1     69        20                1                   6             9       1   
2     46        25                0                   8             4       0   
3     32        24                1                   5             0       0   
4     60        21                0                   9            11       0   

   TempoEspera  DiaSemana_Quarta  DiaSemana_Quinta  DiaSemana_Segunda  \
0           51                 0                 0                  0   
1           32                 0                 0                  0   
2           43                 1                 0                  0   
3           10                 0                 0                  0   
4           63                 0                 0                  0   

   DiaSemana_Sexta

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. Separar as variáveis de entrada (X) e a variável alvo (y)
# Usamos o truque do.split() para evitar que o chat apague os dados
colunas_excluir = 'NoShow,TempoEspera'.split(',')
X = df_encoded.drop(columns=colunas_excluir)
y = df_encoded.NoShow

# 2. Dividir os dados em Treino (80%) e Teste (20%) com semente aleatória para reprodutibilidade
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# 3. Inicializar e treinar o nosso modelo de árvore (Random Forest)
modelo_noshow = RandomForestClassifier(random_state=42)
modelo_noshow.fit(X_train, y_train)

print("--- DIVISÃO CONCLUÍDA ---")
print("Quantidade de dados para Treino (X_train):", X_train.shape)
print("Quantidade de dados para Teste (X_test):", X_test.shape)
print("\nModelo de Previsão de No-Show treinado com sucesso!")

--- DIVISÃO CONCLUÍDA ---
Quantidade de dados para Treino (X_train): (800, 16)
Quantidade de dados para Teste (X_test): (200, 16)

Modelo de Previsão de No-Show treinado com sucesso!


In [15]:
from sklearn.metrics import classification_report, accuracy_score

# 1. Usar o modelo treinado para prever se os 200 pacientes de teste vão faltar ou comparecer
previsoes_noshow = modelo_noshow.predict(X_test)

# 2. Gerar e exibir o relatório completo de métricas
print("--- RELATÓRIO DE DESEMPENHO DA IA (PREVISÃO DE NO-SHOW) ---")
print(classification_report(y_test, previsoes_noshow))

--- RELATÓRIO DE DESEMPENHO DA IA (PREVISÃO DE NO-SHOW) ---
              precision    recall  f1-score   support

           0       0.70      0.87      0.78       134
           1       0.48      0.24      0.32        66

    accuracy                           0.67       200
   macro avg       0.59      0.56      0.55       200
weighted avg       0.63      0.67      0.63       200



In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. O TRUQUE DE OURO: Filtrar apenas os pacientes que compareceram (NoShow == 0)
df_presentes = df_encoded.query("NoShow == 0")

# 2. Separar as variáveis de entrada (X) e a variável alvo (y - TempoEspera)
# Usamos o.split() para que o chat não apague a lista de colunas a excluir
colunas_excluir_reg = "NoShow,TempoEspera".split(",")
X_reg = df_presentes.drop(columns=colunas_excluir_reg)
y_reg = df_presentes.TempoEspera

# 3. Dividir os dados em Treino (80%) e Teste (20%) para a regressão
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, 
    test_size=0.2, 
    random_state=42
)

# 4. Inicializar e treinar os dois modelos concorrentes
modelo_linear = LinearRegression()
modelo_boosting = GradientBoostingRegressor(random_state=42)

modelo_linear.fit(X_train_r, y_train_r)
modelo_boosting.fit(X_train_r, y_train_r)

# 5. Gerar as previsões de tempo no grupo de teste para avaliação
pred_linear = modelo_linear.predict(X_test_r)
pred_boosting = modelo_boosting.predict(X_test_r)

# 6. Calcular as métricas de erro para a Regressão Linear
rmse_linear = np.sqrt(mean_squared_error(y_test_r, pred_linear))
r2_linear = r2_score(y_test_r, pred_linear)

# 7. Calcular as métricas de erro para o Gradient Boosting
rmse_boosting = np.sqrt(mean_squared_error(y_test_r, pred_boosting))
r2_boosting = r2_score(y_test_r, pred_boosting)

print("--- AVALIAÇÃO DOS MODELOS DE REGRESSÃO (TEMPO DE ESPERA) ---")
print(f"Regressão Linear  -> Erro Médio (RMSE): {rmse_linear:.2f} minutos | Ajuste (R²): {r2_linear:.4f}")
print(f"Gradient Boosting -> Erro Médio (RMSE): {rmse_boosting:.2f} minutos | Ajuste (R²): {r2_boosting:.4f}")

--- AVALIAÇÃO DOS MODELOS DE REGRESSÃO (TEMPO DE ESPERA) ---
Regressão Linear  -> Erro Médio (RMSE): 3.44 minutos | Ajuste (R²): 0.9758
Gradient Boosting -> Erro Médio (RMSE): 3.56 minutos | Ajuste (R²): 0.9741


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Definir a grade de parâmetros usando tuplas (parênteses) para o chat não apagar
# Testaremos variações de quantidade de árvores, profundidade e divisão de nós
grade_parametros = dict(
    n_estimators=(50, 100, 150),
    max_depth=(5, 10, None),
    min_samples_split=(2, 5)
)

# 2. Configurar o Grid Search com Validação Cruzada de 5 dobras (cv=5)
# Usamos scoring='f1' para forçar o computador a buscar o equilíbrio entre precisão e recall
busca_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=grade_parametros,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# 3. Executar o treinamento de busca exaustiva (o computador testará todas as combinações)
busca_grid.fit(X_train, y_train)

# 4. Descobrir quais foram os melhores botões de ajuste encontrados pela IA
print("--- RESULTADO DO GRID SEARCH ---")
print("Melhores parâmetros encontrados:", busca_grid.best_params_)
print(f"Melhor pontuação média de F1-Score no treino: {busca_grid.best_score_:.4f}")

# 5. Testar o novo modelo otimizado usando as 200 consultas da gaveta de teste
modelo_otimizado = busca_grid.best_estimator_
novas_previsoes = modelo_otimizado.predict(X_test)

print("\n--- NOVO RELATÓRIO DE DESEMPENHO (MODELO OTIMIZADO) ---")
print(classification_report(y_test, novas_previsoes))